# N072 · 树上路径与信息传递

**目标：** 区分向父亲延伸的路径和已完成的双侧路径。

**先修：** N071, N032。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 根到叶；任意路径；负贡献截断；前缀计数与回溯清理。

**配套讲解来源：** [同名逐章意见](../../comment/072_tree_path_problems.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

树上一大类题目都在问"路径"。但"路径"这个词在不同题里含义差很远：有的题要求从根走到叶（不能半路停）；有的题允许任意两点之间的路径（可以拐弯，但每条边最多走一次）；有的题只允许一直向下走。本章用三个函数覆盖三种形态：`has_path_sum` 问"存不存在一条根到叶的路径，节点值之和恰好等于目标"；`max_path_sum` 问"任意路径的节点值之和最大是多少（节点值可以是负数）"；`path_sum_count` 问"所有只向下走的路径里，和等于目标的有多少条"。

举一个具体到数字的例子。输入这棵树（层序序列 `[-10,9,20,None,None,15,7]`）：

```
        -10
       /    \
      9      20
            /  \
          15    7
```

`max_path_sum` 的输出是 42。为什么？路径 15→20→7 的和是 15+20+7=42，你找不到和更大的路径：把 -10 或 9 掺进来只会让总和变小。注意这条路径在节点 20 处"拐弯"（从左子树上来、再下到右子树），它不经过根——这就是"任意路径"和"根到叶路径"的本质区别，也是本章最核心的新知识点。

## 2. 基础知识：先读懂这些词

- **根到叶路径**：从根出发一直往下走、走到一个叶子（没有孩子的节点）才停的路径。`has_path_sum` 只认这种路径，中途有孩子的节点不算终点。
- **任意路径**：树上任意选两个节点，它们之间那条唯一的连线，可以拐弯。`max_path_sum` 找的是这种，路径的最高点（离根最近的点）可以在树中任何位置。
- **向父亲延伸的路径（单支贡献）与已完成的双侧路径（双支候选）**：这是本章最重要的区分。一个节点如果要向父节点汇报"经过我的最好的路径"，只能带上左、右两条分支中的一条——因为路径不能分叉，带两条就变成"人字形"，没法再往上接。但如果只是在本节点处"结算"一条完整路径（不往上延伸），就可以左右两条都要。
- **负贡献截断**：孩子子树如果总和是负数，把它的值带在身上只会拖累路径和，不如直接砍掉（当作 0）。代码里就是 `max(0, gain[孩子])`。
- **前缀和（prefix sum）**：从根走到当前节点一路加出来的总和。树上任何一段"向下路径"的和 = 两个前缀和之差，这个想法把"数路径"转化成"数前面出现过多少个匹配的前缀"。
- **前缀计数与回溯清理**：用一个计数器记录"当前从根到我的这条祖先链上，每种前缀和出现过几次"。进入一个节点时把它的前缀和加进计数器；离开这棵子树时必须减回去。不清扫的话，左分支的前缀会留在计数器里，被右分支错误地匹配，拼出一条实际不存在的"跨分支假路径"。
- **后序遍历（postorder）**：先左右孩子、后自己。上一章 N071 讲过，本章 `max_path_sum` 继续用它，保证算父节点时两个孩子的贡献已经算好。

## 3. 思路推导：从小例子开始

### 1. `has_path_sum`：根到叶，一条道走到黑

- **Step 1**：沿着树往下走，手里始终带着"从根到当前节点的累加和"。
- **Step 2**：每到一个节点，先加上自己的值；如果自己是叶子（左右都是空），就检查累加和是否等于 target。
- **Step 3**：手算一遍。树是 `1` 带左孩子 `2`，target=1：走到节点 1，累加和 1，但它有孩子，不算到达终点；走到节点 2，累加和 3，是叶子，3≠1；没有别的路了，答案是不存在。这就是 “自动测试”部分 第三条断言的来历。

### 2. `max_path_sum`：每个节点做一次"拐弯结算"

- **Step 1**：对每个节点问两个问题：第一，"只从我这向上延伸"最好能贡献多少（单支贡献，只能选左或右一边）；第二，"以我为最高点、左右都下探"的路径和是多少（双支候选）。
- **Step 2**：单支贡献 = 自己的值 + max(0, 左孩子贡献, 右孩子贡献)——孩子是负贡献就当 0（砍掉）。双支候选 = 自己的值 + max(0,左) + max(0,右)，左右都可以要。
- **Step 3**：手算 notebook “运行示例”部分 用的小树 `[-10,9,20,None,None,15,7]`，后序顺序是 9, 15, 7, 20, -10：

| 节点 | 可向上传的单支贡献 | 在此结束的双支候选 |
|------|--------------------|--------------------|
| 9    | 9                  | 9                  |
| 15   | 15                 | 15                 |
| 7    | 7                  | 7                  |
| 20   | 20+max(15,7)=35    | 20+15+7=42         |
| -10  | -10+max(9,35)=25   | -10+9+35=34        |

逐格读：节点 9、15、7 都是叶子，单支贡献就是自己的值；节点 20 的左孩子贡献 15、右孩子贡献 7，单支只能挑大的一边（20+15=35），双支两边都要（20+15+7=42）；节点 -10 的左孩子能传 9、右孩子能传 35，单支只允许选一边，挑大的 35，得 -10+35=25，而双支候选可以把两边都算上，得 -10+9+35=34。这一行正是"单支只能选一边"最直观的体现：25 和 34 的差别，就是"要不要同时带上左右两条胳膊"。建议你亲手运行 “运行示例”部分，把输出的表格和上表逐格核对。

- **Step 4**：全局答案 = 所有节点双支候选的最大值 = max(9,15,7,42,34) = 42。

### 3. `path_sum_count`：前缀和 + 数祖先

- **Step 1**：把"从祖先 a 向下走到当前节点 c 的路径和"改写成"两个前缀和之差"：`prefix(c) - prefix(a的父亲)`。所以和等于 target 等价于 `prefix(a的父亲) = prefix(c) - target`。
- **Step 2**：往下走时带一个计数器，记录当前祖先链上每种前缀和出现的次数；每到一个节点，先问"我之前有多少个祖先的前缀和恰好等于 total-target"，这个数就是以我为终点、和为 target 的路径条数。
- **Step 3**：计数器初始 `{0:1}`，代表"根之前的前缀和 0 出现过一次"，这样"从根出发的整条路径"也能被数到。
- **Step 4**：关键纪律——离开一个子树时把它的前缀和从计数器里减掉（进入加一、退出减一）。这样计数器里永远只有"从根到当前节点"这一条链上的值，不可能匹配到别的分支。
- **Step 5**：给 `path_sum_count` 补一个手算演示。用测试里那棵大树 `[10,5,-3,3,2,None,11,3,-2,None,1]` 的左半边（10、5、-3、11）就够看，target=8。各节点的前缀和依次是：10→15（到 5）、7（到 -3）、18（到 11）。逐个节点走：到 10 时 total=10，问计数器"有几个 10-8=2"，答案 0，登记 10；到 5 时 total=15，问"有几个 7"，0，登记 15；到 -3 时 total=7，问"有几个 -1"，0，登记 7；到 11 时 total=18，问"有几个 10"，1——根的前缀和 10 在账上，意味着"从根的下一个节点（-3）一路走到 11"这段的和是 18-10=8，即 -3→11 这条路径，answer 加一。你看，"两前缀之差就是一段路径和"在这格被演出来了。

## 4. 核心代码：has_path_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def has_path_sum(root, target):
    stack = [(root, 0)] if root else []
    while stack:
        node, prefix = stack.pop()
        total = prefix + node.val
        if not node.left and (not node.right) and (total == target):
            return True
        if node.right:
            stack.append((node.right, total))
        if node.left:
            stack.append((node.left, total))
    return False
```

### 逐段读懂代码

“分段实现”部分 开头的 `TreeNode`、`tree_from_level`、`_postorder_nodes` 与 N071 完全相同（建树工具和显式后序栈，上一章已逐行讲过），这里直接看三个新函数。

### 1. `has_path_sum`：带累加和的深度优先

```python
def has_path_sum(root, target):
    stack = [(root, 0)] if root else []
    while stack:
        node, prefix = stack.pop()
        total = prefix + node.val
        if not node.left and (not node.right) and (total == target):
            return True
        if node.right:
            stack.append((node.right, total))
        if node.left:
            stack.append((node.left, total))
    return False
```

栈里每项是 `(节点, 从根到这里的累加和)`。第一行 `stack=[(root,0)] if root else []` 是紧凑写法：树为空时给一个空栈，循环一次不跑，直接返回 `False`。弹出节点后 `total=prefix+node.val` 把自己的值累加上；`not node.left and not node.right` 判断"是叶子"，叶子上检查 `total==target`。孩子存在才压栈，`None` 不会被压进去，所以循环里不用判空。只要找到一条满足就立刻返回 `True`，全部扫完没找到才返回 `False`。

### 2. `max_path_sum`：后序 + 贡献字典 + 全局最优

```python
def max_path_sum(root):
    if not root:
        raise ValueError('nonempty tree required')
    gain = {None: 0}
    best = float('-inf')
    for node in _postorder_nodes(root):
        left = max(0, gain[node.left])
        right = max(0, gain[node.right])
        best = max(best, node.val + left + right)
        gain[node] = node.val + max(left, right)
    return best
```

第一行不是特判返回值，而是直接抛异常：这题的定义要求路径至少含一个节点，空树没有合法答案，与其悄悄返回一个误导性的数字，不如报错把问题暴露出来。`gain` 字典存"每个节点能向上传的单支贡献"，`gain[None]=0` 是空树地基。循环体两行就是第三节的推导：先算左右孩子截断后的贡献，`best` 用双支候选 `node.val+left+right` 刷新（这一步对应"以我为最高点结算"），然后 `gain[node]=node.val+max(left,right)` 只挑一边存起来向上传（这一步对应"路径不能分叉"）。`best` 初值是负无穷 `float('-inf')`，保证全负树也会老老实实选出最大的那个单节点（见测试第二条）。

### 3. `path_sum_count`：进栈加一、出栈减一

```python
def path_sum_count(root, target):
    counts = Counter({0: 1})
    answer = 0
    stack = [(root, 0, False)]
    while stack:
        node, prefix, leaving = stack.pop()
        if not node:
            continue
        if leaving:
            counts[prefix] -= 1
            if counts[prefix] == 0:
                del counts[prefix]
            continue
        total = prefix + node.val
        answer += counts[total - target]
        counts[total] += 1
        stack.append((node, total, True))
        stack.append((node.right, total, False))
        stack.append((node.left, total, False))
    return answer
```

栈里每项是 `(节点, 到该节点的前缀和, 是否是"离开"标记)`。正常进入一个节点（`leaving=False`）时做三件事：算出 `total`；`answer+=counts[total-target]`——查询祖先链上有多少个前缀和能和当前节点拼出 target，这一步顺便把"从根开始的整条路径"也数到（靠初始的 `{0:1}`）；`counts[total]+=1` 把自己的前缀和登记进计数器。然后压三个任务：先压"离开时清理"，再压右孩子、左孩子。栈是后进先出，所以实际执行顺序是：左子树全部处理完 → 右子树全部处理完 → 才轮到 `(node,total,True)` 这个离开标记。离开时 `counts[prefix]-=1` 把登记撤销；如果减到 0 就顺手 `del` 掉这个键，让计数器始终保持"只含当前祖先路径"这个状态。如果删掉这几行清理代码会怎样？左分支的前缀和会一直留在计数器里，右分支的节点能匹配到它，数出一条实际不存在的"从左分支跳到右分支"的假路径——这正是练习 2 要你亲手构造的场景。

## 5. 分段实现：按顺序运行

**本章接口：** `has_path_sum(root, target)`、`max_path_sum(root)`、`path_sum_count(root, target)`

### TreeNode

In [1]:
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

### tree_from_level

In [2]:
def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

### _postorder_nodes

In [3]:
def _postorder_nodes(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not node:
            continue
        if done:
            yield node
        else:
            stack.append((node, True))
            stack.append((node.right, False))
            stack.append((node.left, False))

### 导入本章使用的库

In [4]:
from collections import Counter

### has_path_sum

In [5]:
def has_path_sum(root, target):
    stack = [(root, 0)] if root else []
    while stack:
        node, prefix = stack.pop()
        total = prefix + node.val
        if not node.left and (not node.right) and (total == target):
            return True
        if node.right:
            stack.append((node.right, total))
        if node.left:
            stack.append((node.left, total))
    return False

### max_path_sum

In [6]:
def max_path_sum(root):
    if not root:
        raise ValueError('nonempty tree required')
    gain = {None: 0}
    best = float('-inf')
    for node in _postorder_nodes(root):
        left = max(0, gain[node.left])
        right = max(0, gain[node.right])
        best = max(best, node.val + left + right)
        gain[node] = node.val + max(left, right)
    return best

### path_sum_count

In [7]:
def path_sum_count(root, target):
    counts = Counter({0: 1})
    answer = 0
    stack = [(root, 0, False)]
    while stack:
        node, prefix, leaving = stack.pop()
        if not node:
            continue
        if leaving:
            counts[prefix] -= 1
            if counts[prefix] == 0:
                del counts[prefix]
            continue
        total = prefix + node.val
        answer += counts[total - target]
        counts[total] += 1
        stack.append((node, total, True))
        stack.append((node.right, total, False))
        stack.append((node.left, total, False))
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [8]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([-10,9,20,None,None,15,7]); gain={None:0}; rows=[]
for n in _postorder_nodes(r):
    l=max(0,gain[n.left]); rr=max(0,gain[n.right]); gain[n]=n.val+max(l,rr)
    rows.append((n.val,gain[n],n.val+l+rr))
show_table(['节点','可向上传的单支贡献','在此结束的双支候选'],rows)

节点,可向上传的单支贡献,在此结束的双支候选
9,9,9
15,15,15
7,7,7
20,35,42
-10,25,34


## 7. 正确性、适用条件与复杂度

最大路径按唯一最高节点分类，向上返回值与全局候选承担不同语义；全负树仍至少取一个节点。前缀频率只含当前祖先路径，进入加一、退出减一维持范围。

**代价：** O(n)平均时间；最大和高度表O(n)空间；前缀计数工作空间O(h)，极端h=n。

### 展开理解

**`max_path_sum` 为什么是对的？** 任意一条路径在树上都有唯一一个"最高点"（离根最近的节点），这条路径从最高点的左子树某处上来、穿过最高点、再下到右子树某处（两段中允许有一段为空）。所以所有可能的最优路径，可以按最高点是哪个节点分成若干组，每组里"最好的一条"恰好就是我们在该节点算的双支候选：左右两段各自要"从最高点向下能得到的最大和"，负贡献截断保证孩子若帮倒忙就干脆不要。我们对每个节点都算了候选并取全局最大，任何一条路径都属于某一组，所以最优解一定被算到。向上传的 `gain` 只带一边，是因为带两条的"人字形"没法再接父节点，硬传上去就不再是路径了。

**全负树为什么答案是 -2？** `best` 从负无穷起步，每个节点的候选都要和它比，所以至少会选中一个节点。树 `[-3,-2,-5]` 三个候选是 -3、-2（单节点）、-5（单节点）以及它们的组合（都更小），最大是 -2。"路径至少含一个节点"是题目定义，写成 `max(0, ...)` 返回 0 反而是错的。

**`path_sum_count` 为什么是对的？** 关键不变量：站在任何节点上时，计数器里装的恰好是"从根到当前节点这条链上，每个位置的前缀和的出现次数"，不多不少。进入时加一、离开时减一维持了这个状态（这正是"回溯"）；由于任何向下路径的终点和起点都在同一条根链上，`total-target` 的查询就恰好数出所有以当前节点为终点、和为 target 的路径，既不漏也不多。

**复杂度**：三个函数每个节点都只进出栈常数次、查表常数次，所以时间都是 O(n)（`Counter` 单次操作平均 O(1)）。n=10⁵ 时各自就是十万级别的常数操作，瞬间跑完。空间方面：`max_path_sum` 的 `gain` 字典要给每个节点存一个数，是 O(n)；`path_sum_count` 的计数器只装当前一条祖先链，是 O(h)（h 是树高），树退化成链时 h=n 达到最坏；`has_path_sum` 的栈同理也是 O(h)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `r=tree_from_level([-10,9,20,None,None,15,7]); assert max_path_sum(r)==42`：标准用例，测的是"最优路径拐弯在中间节点 20、且不经过根"这个核心能力。
- `assert max_path_sum(tree_from_level([-3,-2,-5]))==-2`：全负树特殊值。它检验 `best` 的负无穷初值：答案必须选中最大的单个节点 -2，而不是被错误地截断成 0。
- `assert not has_path_sum(TreeNode(1,TreeNode(2)),1)`：边界语义检验。累加和 1 恰好等于 target，但节点 1 有孩子、不是叶，这条半路停下的路径不算数，必须返回假。它防止你把"是叶子"这个条件漏掉。
- `r=tree_from_level([10,5,-3,3,2,None,11,3,-2,None,1]); assert path_sum_count(r,8)==3`：这棵树上和为 8 的向下路径恰好有三条：5→3、5→2→1、-3→11。它同时覆盖了"从根出发""中间拐点""右单边"各种形态。
- `for target in range(-5,16)` 的对拍循环：对 21 个不同的 target，用 `from_start` 这个笨办法（从每个节点出发向下穷举所有路径）算出答案，和本章的 `path_sum_count` 逐一比对。两种方法思路完全不同，能全部一致，说明前缀计数实现没有串线。
- `assert path_sum_count(None,0)==0`：空树边界。空树一条路径都没有，target 是 0 也不能凭空数出一条来。

全部断言通过后打印 `N072: 所有本章断言通过`。四组用例各自的分工一目了然：42 测拐弯、-2 测全负、False 测叶子判定、对拍测前缀计数不串线。

In [9]:
r = tree_from_level([-10, 9, 20, None, None, 15, 7])

assert max_path_sum(r) == 42

assert max_path_sum(tree_from_level([-3, -2, -5])) == -2

assert not has_path_sum(TreeNode(1, TreeNode(2)), 1)

r = tree_from_level([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1])

assert path_sum_count(r, 8) == 3

def from_start(node, target):
    if not node:
        return 0
    return int(node.val == target) + from_start(node.left, target - node.val) + from_start(node.right, target - node.val)

for target in range(-5, 16):
    assert path_sum_count(r, target) == sum((from_start(n, target) for n in _postorder_nodes(r)))

assert path_sum_count(None, 0) == 0

print('N072: 所有本章断言通过')

N072: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造不能把双臂路径继续向上传递的反例。

**练习 2：** 清理前缀频次防止跨分支串线。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（构造"双臂路径不能向上传"的反例）**：提示——先想清楚命题："如果把双支候选（左右都要的那份）当成单支贡献传给父亲，答案会错"。你要造一棵树，让某个节点的双臂拼起来很大、但它一往上接就变成假路径。方向：让拐弯点的父亲连着一个负得很多或者结构特殊的节点，手算"错误版本"的返回值，和 `max_path_sum` 的正确答案对比，指出错误版本把哪条不存在的路径算进去了。比如把 “运行示例”部分 那棵树的 -10 换成更大的正数试试，看错误实现会不会把 15→20→7 再拐上去。
- **练习 2（清理前缀频次，防止跨分支串线）**：提示——把 `path_sum_count` 里 `if leaving:` 分支整段删掉，得到一个"忘记打扫"的版本，然后专门构造一棵树：左分支和右分支各自的前缀和，恰好能拼出一个等于 target 但实际不存在的组合。方向：让根的左子树里某个前缀和等于 `右子树某节点前缀和 - target`，跑"脏版本"看它多数了几条，再用 “自动测试”部分 的对拍循环抓出第一个不一致的 target。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [10]:
# 完整实现：本单元格不依赖其他单元格。
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

def _postorder_nodes(root):
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not node:
            continue
        if done:
            yield node
        else:
            stack.append((node, True))
            stack.append((node.right, False))
            stack.append((node.left, False))

from collections import Counter

def has_path_sum(root, target):
    stack = [(root, 0)] if root else []
    while stack:
        node, prefix = stack.pop()
        total = prefix + node.val
        if not node.left and (not node.right) and (total == target):
            return True
        if node.right:
            stack.append((node.right, total))
        if node.left:
            stack.append((node.left, total))
    return False

def max_path_sum(root):
    if not root:
        raise ValueError('nonempty tree required')
    gain = {None: 0}
    best = float('-inf')
    for node in _postorder_nodes(root):
        left = max(0, gain[node.left])
        right = max(0, gain[node.right])
        best = max(best, node.val + left + right)
        gain[node] = node.val + max(left, right)
    return best

def path_sum_count(root, target):
    counts = Counter({0: 1})
    answer = 0
    stack = [(root, 0, False)]
    while stack:
        node, prefix, leaving = stack.pop()
        if not node:
            continue
        if leaving:
            counts[prefix] -= 1
            if counts[prefix] == 0:
                del counts[prefix]
            continue
        total = prefix + node.val
        answer += counts[total - target]
        counts[total] += 1
        stack.append((node, total, True))
        stack.append((node.right, total, False))
        stack.append((node.left, total, False))
    return answer

# 示例与回归测试
r = tree_from_level([-10, 9, 20, None, None, 15, 7])

assert max_path_sum(r) == 42

assert max_path_sum(tree_from_level([-3, -2, -5])) == -2

assert not has_path_sum(TreeNode(1, TreeNode(2)), 1)

r = tree_from_level([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1])

assert path_sum_count(r, 8) == 3

def from_start(node, target):
    if not node:
        return 0
    return int(node.val == target) + from_start(node.left, target - node.val) + from_start(node.right, target - node.val)

for target in range(-5, 16):
    assert path_sum_count(r, target) == sum((from_start(n, target) for n in _postorder_nodes(r)))

assert path_sum_count(None, 0) == 0

print('N072: 所有本章断言通过')

N072: 所有本章断言通过


## 11. 代表题与迁移

- **112. Path Sum**：判断根到叶路径和，直接对应 `has_path_sum`，练的是"累加和随身带 + 叶子判定"。
- **113. Path Sum II**：把满足条件的根到叶路径全部列出来，是 112 的加强版，练的是"在同样的走法上多记录一条路径列表，回退时弹栈"。
- **124. Binary Tree Maximum Path Sum**：任意路径最大和，直接对应 `max_path_sum`，练的正是"单支贡献与双支候选分离 + 负贡献截断"这两个本章核心。
- **437. Path Sum III**：数向下路径条数，直接对应 `path_sum_count`，练的是"前缀和计数 + 进入登记/离开撤销"的回溯纪律。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。